(5 pts) In this problem, you will train a Vision Transformer (ViT)-style model entirely from
scratch. Your code should have the full pipeline: data loading, model definition, training,
evaluation, and analysis. Create a single notebook named ViT training.ipynb that is
runnable end-to-end on Google Colab (free tier).
(a) Dataset Selection. Choose any two image-classification datasets, each containing
at least 10 unique classes (e.g. CIFAR-10, CIFAR-100, STL-10, EuroSAT, Oxford-
Pets, Flowers-102, Food-101, etc.) Please don’t use MNIST and its variants (Fashion-
MNIST, KMNIST, EMNIST, etc.) as these datasets are too simple to meaningfully
exercise a transformer architecture.
(b) Model. Design and implement a ViT-like architecture. Your model must include, at
minimum:
• A patch-embedding layer,
• At least two transformer encoder blocks (multi-head self-attention + feed-forward
network),
• Positional encoding (learned or fixed),
• A classification head.
2
You are free to experiment with patch size, embedding dimension, number of heads,
depth, dropout, and any other architectural choices. Hybrid designs that incorporate
convolutional components (e.g. convolutional patch embeddings) are also permitted.
(c) Training. Train your model on both datasets separately using cross-entropy loss. Track
and record training loss, training accuracy, and test/validation accuracy per epoch. Your
test accuracy must be ≥ 40% on both datasets.
(d) Robustness to Domain Shifts. After training, evaluate your model’s robustness by
adding Gaussian noise/blur to the test images at various noise levels. Report accuracy
at each noise level for both datasets. Discuss how performance degrades and whether
the two datasets exhibit different sensitivity to noise.
(e) Plots & Discussion. Include the following plots for each dataset:
• Training loss vs. epoch,
• Training accuracy and test accuracy vs. epoch (on the same axes),
• Test accuracy vs. Gaussian noise level σ.
From your training curves, state whether the model is over-fitted, under-fitted, or well-
fitted, and explain your reasoning. Compare behaviour across the two datasets

## 0) Colab Setup and Reproducibility

### What to implement in the next code block

- Install/import required libraries (`torch`, `torchvision`, `numpy`, `matplotlib`, optionally `tqdm`).
- Detect device (`cuda` if available).
- Set global random seed for reproducibility.
- Print torch version + device info.

### Notes

- Use mixed precision (`torch.cuda.amp`) only if comfortable; not required.
- Keep all constants (batch size, epochs, LR, image size, patch size, etc.) in one config block so experiments are easy to change.
- Add a short runtime estimate note for Colab free tier.

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt 
import tqdm

device = torch.device("cuda:0") if torch.cuda.is_available() else "cpu"
np.random.seed(42)

BATCH_SIZE = 24
EPOCHS = 100
LR = 1e-3
IMAGE_SIZE = (224, 224)

## (a) Dataset Selection and Data Loading

Choose two datasets with >= 10 classes (not MNIST variants). Recommended for Colab speed:

- **Dataset A**: CIFAR-10
- **Dataset B**: STL-10 (or CIFAR-100 if preferred)

### What to implement in the next code blocks

1. **Dataset choice markdown cell**
   - State why these two datasets were chosen (difficulty/domain differences).

2. **Transforms and dataloaders**
   - Define train and test transforms.
   - Ensure image size is consistent with your ViT patching strategy.
   - Create train/test datasets and dataloaders for A and B.

3. **Sanity checks**
   - Print number of classes, train/test sizes, and one batch shape.
   - Visualize a few sample images + labels for each dataset.

### Suggested transform policy

- Train: resize (if needed), random crop/flip, normalize.
- Test: deterministic resize + normalize.

### Important

- Keep dataloader code reusable: one helper function per dataset to avoid duplication.

Choosing CIFAR-10 and STL-10 because a multiclass classification among 10 classes could be compared domain sensitivity



In [ ]:
from torch import TensorDataset, DataLoader

def get_dataloader(dataset_name, img_size = 64, batch_size = 128):

    if dataset_name == 'CIFAR-10':
        train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
        test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms.ToTensor())
    elif dataset_name == 'STL-10':
        train_dataset = torchvision.datasets.STL10(root='./data', split='train', download=True, transform=transforms.ToTensor())
        test_dataset = torchvision.datasets.STL10(root='./data', split='test', download=True, transform=transforms.ToTensor())
    else:
        raise ValueError(f"Dataset {dataset_name} not supported")
        

    train_loader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size, shuffle=False, num_workers=4, pin_memory=True)
    num_classes = len(train_dataset.classes)
    class_names = train_dataset.classes

    return train_loader, test_loader, num_classes, class_names


In [ ]:
def transform_policy(img_size = 64, set = 'train'):
    if set == 'train':
        return transforms.Compose([
            transforms.Resize(img_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
    else:
        return transforms.Compose([
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

## (b) ViT-Style Model Definition

### Minimum architecture requirements to satisfy

- Patch embedding layer.
- Positional encoding (learned or fixed).
- >= 2 transformer encoder blocks.
- Classification head.

### Suggested implementation structure

Create classes/functions in this order:

1. `PatchEmbedding`
   - Input: `B x C x H x W`
   - Output: token sequence `B x N x D`
   - Use either:
     - `nn.Conv2d` with `kernel_size=patch_size`, `stride=patch_size`, or
     - manual flatten + linear projection.

2. `TransformerEncoderBlock`
   - LayerNorm -> Multi-head self-attention -> residual.
   - LayerNorm -> MLP/FFN -> residual.
   - Include dropout.

3. `VisionTransformer`
   - Create class token (optional but standard).
   - Add positional embeddings.
   - Stack encoder blocks.
   - Pooling/class token selection.
   - Final linear classifier.

### Hyperparameter guidance (starter)

- image_size: 32 or 64
- patch_size: 4 or 8
- embed_dim: 128-256
- depth: 4-6 (minimum 2)
- num_heads: 4-8
- mlp_ratio: 2-4
- dropout: 0.1

### Verification checklist

- Print model summary and parameter count.
- Run one forward pass on a dummy batch and verify output shape is `B x num_classes`.

## (c) Training and Evaluation Pipeline

### What to implement in the next code blocks

1. **Train/eval functions**
   - `train_one_epoch(model, loader, optimizer, criterion, device)`
   - `evaluate(model, loader, criterion, device)`
   - Return loss and accuracy.

2. **Experiment runner**
   - `run_experiment(dataset_name, model_config, train_loader, test_loader, num_classes)`
   - Initialize model from scratch.
   - Use cross-entropy loss.
   - Train for `E` epochs and log:
     - train loss,
     - train acc,
     - test acc.

3. **Run for both datasets**
   - Train on Dataset A.
   - Train on Dataset B.
   - Save metrics history in dicts for plotting.

### Optimization guidance

- Optimizer: `AdamW`.
- LR: start around `1e-3` (tune if unstable).
- Weight decay: ~`1e-4`.
- Optional scheduler: cosine or step decay.
- Epoch target for Colab: 15-40 depending on dataset and model size.

### Requirement reminder

- You must report >= 40% test accuracy for both datasets.
- If below threshold, tune depth, embed dim, augmentation, epochs, or LR schedule.

## (d) Robustness to Domain Shift (Gaussian Corruption)

### Goal

Evaluate trained models under increasing corruption and compare degradation across datasets.

### What to implement

1. Define sigma levels, e.g.:
   - `[0.0, 0.05, 0.1, 0.15, 0.2, 0.3]`

2. Build corrupted test transform/function:
   - Add Gaussian noise with each sigma, and/or
   - Apply Gaussian blur with configurable kernel/sigma.

3. For each sigma:
   - Evaluate accuracy on corrupted test set for Dataset A and Dataset B.
   - Store results in arrays/lists for plotting.

### Implementation tips

- Keep corruption deterministic per run by controlling seed if needed.
- Ensure corruption is applied on test images only.
- Use same trained weights as standard test evaluation.

### Analysis prompts

- Which dataset degrades faster with sigma?
- Does accuracy drop smoothly or sharply?
- Is there a robustness gap even if clean accuracy is similar?

## (e) Plots and Discussion

Create the following plots for **each dataset**:

1. Training loss vs epoch.
2. Training accuracy and test accuracy vs epoch (same axes).
3. Test accuracy vs Gaussian noise level sigma.

### What to include in written discussion

- Fit diagnosis for each dataset:
  - **Overfitting**: train acc high, test acc stalls/drops.
  - **Underfitting**: both train/test low.
  - **Well-fitted**: train and test both improve with modest gap.
- Compare Dataset A vs B:
  - Which is harder to optimize?
  - Which is more robust to corruption?
  - How class diversity and image characteristics may explain behavior.

### Reporting checklist

- Final clean test accuracy (A and B).
- Best epoch and best test accuracy for each dataset.
- Noise robustness table or printed list of `(sigma, acc)` for each dataset.
- 1-2 paragraph conclusion summarizing key insights.

## Suggested Notebook Execution Order

1. Setup/imports/seed/config.
2. Dataset loading + quick sample visualization.
3. Model class definitions.
4. Train/eval utilities.
5. Train + evaluate on Dataset A.
6. Train + evaluate on Dataset B.
7. Robustness evaluation (noise/blur) for A and B.
8. Plotting section.
9. Final discussion section.

Keep each section self-contained and avoid hidden state dependencies so reruns work cleanly on Colab.

## Quick Grading Self-Check (before submission)

- [ ] Notebook name is exactly `ViT training.ipynb`.
- [ ] End-to-end Colab runnable without manual patching.
- [ ] Two datasets used, each with >= 10 classes.
- [ ] ViT has patch embedding + positional encoding + >= 2 encoder blocks + classifier.
- [ ] Cross-entropy training done separately on both datasets.
- [ ] Tracked train loss, train acc, test acc per epoch.
- [ ] >= 40% test accuracy achieved on both datasets.
- [ ] Gaussian corruption robustness evaluated at multiple sigma levels.
- [ ] All required plots included.
- [ ] Discussion includes fit diagnosis and cross-dataset comparison.